In [2]:
import pandas as pd
import numpy as np
import os
import glob

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Environment ready!")

Environment ready!


In [3]:
DATA_PATH = "data/raw/"

files = sorted(glob.glob(os.path.join(DATA_PATH, "*.csv")))

print(f"CSV files found: {len(files)}")
print("\n".join(os.path.basename(file) for file in files))

CSV files found: 18
account_status_history.csv
accounts.csv
agent_sessions.csv
agents.csv
borrowers.csv
call_attempts.csv
call_dispositions.csv
calls.csv
campaigns.csv
complaints.csv
daily_targeting.csv
data_dictionary.csv
field_visits.csv
payments.csv
promises_to_pay.csv
sms_events.csv
vendor_telephony.csv
whatsapp_events.csv


In [4]:
data = {}

for file in files:
    table_name = os.path.splitext(os.path.basename(file))[0]
    data[table_name] = pd.read_csv(file)

print("All datasets loaded successfully.\n")

for name, df in data.items():
    print(f"{name:<25} {df.shape[0]:>10,} rows × {df.shape[1]:>3} columns")

All datasets loaded successfully.

account_status_history        60,000 rows ×   8 columns
accounts                      30,000 rows ×  11 columns
agent_sessions                15,000 rows ×   7 columns
agents                        30,000 rows ×   8 columns
borrowers                     30,600 rows ×   8 columns
call_attempts                120,000 rows ×   9 columns
call_dispositions             35,000 rows ×   8 columns
calls                         91,350 rows ×  11 columns
campaigns                        120 rows ×   7 columns
complaints                     8,000 rows ×   9 columns
daily_targeting               45,000 rows ×   7 columns
data_dictionary                  143 rows ×   3 columns
field_visits                  25,000 rows ×  10 columns
payments                      25,500 rows ×   9 columns
promises_to_pay               18,000 rows ×   9 columns
sms_events                    45,000 rows ×   8 columns
vendor_telephony                  15 rows ×   6 columns
whatsapp_even

In [5]:
inventory = []

for name, df in data.items():
    inventory.append({
        "table": name,
        "rows": len(df),
        "columns": len(df.columns),
        "exact_duplicates": df.duplicated().sum(),
        "missing_cells": df.isna().sum().sum(),
        "unique_rows": len(df.drop_duplicates())
    })

inventory_df = (
    pd.DataFrame(inventory)
    .sort_values("rows", ascending=False)
    .reset_index(drop=True)
)

inventory_df

,table,rows,columns,exact_duplicates,missing_cells,unique_rows
0,call_attempts,120000,9,0,2400,120000
1,calls,91350,11,1271,1827,90079
2,whatsapp_events,60600,8,600,0,60000
3,account_status_history,60000,8,0,0,60000
4,sms_events,45000,8,0,0,45000
5,daily_targeting,45000,7,0,0,45000
6,call_dispositions,35000,8,0,0,35000
7,borrowers,30600,8,600,1509,30000
8,accounts,30000,11,0,455,30000
9,agents,30000,8,0,0,30000


In [6]:
os.makedirs("output/data_quality", exist_ok=True)

inventory_df.to_csv(
    "output/data_quality/raw_data_inventory.csv",
    index=False
)

print("Raw data inventory saved successfully!")

Raw data inventory saved successfully!


Understand Table Structure & Relationships


In [7]:
# Create a detailed table structure summary

table_structure = []

for name, df in data.items():
    table_structure.append({
        "table": name,
        "rows": len(df),
        "columns": len(df.columns),
        "column_names": ", ".join(df.columns.tolist())
    })

table_structure_df = pd.DataFrame(table_structure)

table_structure_df

,table,rows,columns,column_names
0,account_status_history,60000,8,"history_id, account_id, borrower_id, event_at,..."
1,accounts,30000,11,"account_id, borrower_id, loan_type, principal_..."
2,agent_sessions,15000,7,"session_id, agent_id, login_at, channel, devic..."
3,agents,30000,8,"agent_id, employee_code, agent_name, vendor_id..."
4,borrowers,30600,8,"borrower_id, name, phone, email, city, created..."
5,call_attempts,120000,9,"attempt_id, account_id, borrower_id, event_at,..."
6,call_dispositions,35000,8,"disposition_id, account_id, borrower_id, event..."
7,calls,91350,11,"call_id, account_id, borrower_id, event_at, ag..."
8,campaigns,120,7,"campaign_id, campaign_name, channel, strategy_..."
9,complaints,8000,9,"complaint_id, account_id, borrower_id, event_a..."


In [8]:
schema_summary = []

for name, df in data.items():
    for column in df.columns:
        schema_summary.append({
            "table": name,
            "column": column,
            "data_type": str(df[column].dtype),
            "missing_values": df[column].isna().sum(),
            "unique_values": df[column].nunique()
        })

schema_df = pd.DataFrame(schema_summary)

schema_df

,table,column,data_type,missing_values,unique_values
0,account_status_history,history_id,str,0,60000
1,account_status_history,account_id,str,0,25999
2,account_status_history,borrower_id,str,0,11916
3,account_status_history,event_at,str,0,59898
4,account_status_history,status,str,0,7
...,...,...,...,...,...
141,whatsapp_events,event_at,str,0,59892
142,whatsapp_events,message_id,str,0,34831
143,whatsapp_events,event_type,str,0,6
144,whatsapp_events,template_code,str,0,5


In [9]:
data["data_dictionary"].head(20)

,dataset,column,dtype
0,borrowers,borrower_id,object
1,borrowers,name,object
2,borrowers,phone,object
3,borrowers,email,object
4,borrowers,city,object
5,borrowers,created_at,datetime64[ns]
6,borrowers,updated_at,datetime64[ns]
7,borrowers,state,object
8,accounts,account_id,object
9,accounts,borrower_id,object


In [10]:
print(data["data_dictionary"].columns.tolist())

['dataset', 'column', 'dtype']


In [11]:
data["data_dictionary"].info()

<class 'pandas.DataFrame'>
RangeIndex: 143 entries, 0 to 142
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   dataset  143 non-null    str  
 1   column   143 non-null    str  
 2   dtype    143 non-null    str  
dtypes: str(3)
memory usage: 3.5 KB


In [12]:
# Summary of each dataset and its columns

for name, df in data.items():
    if name != "data_dictionary":
        print("\n" + "=" * 90)
        print(f"DATASET: {name.upper()}")
        print("=" * 90)
        
        print(f"Rows: {len(df):,}")
        print(f"Columns: {len(df.columns)}")
        print("\nColumn names:")
        print(df.columns.tolist())


DATASET: ACCOUNT_STATUS_HISTORY
Rows: 60,000
Columns: 8

Column names:
['history_id', 'account_id', 'borrower_id', 'event_at', 'status', 'changed_by', 'source', 'recorded_at']

DATASET: ACCOUNTS
Rows: 30,000
Columns: 11

Column names:
['account_id', 'borrower_id', 'loan_type', 'principal_amount', 'outstanding_amount', 'dpd', 'risk_segment', 'status', 'opened_at', 'timezone', 'schema_version']

DATASET: AGENT_SESSIONS
Rows: 15,000
Columns: 7

Column names:
['session_id', 'agent_id', 'login_at', 'channel', 'device_id', 'timezone', 'logout_at']

DATASET: AGENTS
Rows: 30,000
Columns: 8

Column names:
['agent_id', 'employee_code', 'agent_name', 'vendor_id', 'team', 'status', 'joined_at', 'updated_at']

DATASET: BORROWERS
Rows: 30,600
Columns: 8

Column names:
['borrower_id', 'name', 'phone', 'email', 'city', 'created_at', 'updated_at', 'state']

DATASET: CALL_ATTEMPTS
Rows: 120,000
Columns: 9

Column names:
['attempt_id', 'account_id', 'borrower_id', 'event_at', 'call_id', 'agent_id', 'att

In [13]:
primary_key_candidates = []

for name, df in data.items():
    if name == "data_dictionary":
        continue
    
    for column in df.columns:
        missing = df[column].isna().sum()
        unique = df[column].nunique()
        
        if missing == 0 and unique == len(df):
            primary_key_candidates.append({
                "dataset": name,
                "candidate_key": column,
                "rows": len(df),
                "unique_values": unique
            })

primary_key_candidates_df = pd.DataFrame(primary_key_candidates)

primary_key_candidates_df

,dataset,candidate_key,rows,unique_values
0,account_status_history,history_id,60000,60000
1,accounts,account_id,30000,30000
2,agent_sessions,session_id,15000,15000
3,call_attempts,attempt_id,120000,120000
4,call_dispositions,disposition_id,35000,35000
5,campaigns,campaign_id,120,120
6,campaigns,start_at,120,120
7,campaigns,end_at,120,120
8,complaints,complaint_id,8000,8000
9,daily_targeting,target_id,45000,45000


Common Relationship Columns


In [14]:
id_columns = []

for name, df in data.items():
    for column in df.columns:
        if (
            column.endswith("_id")
            or column in ["account_id", "borrower_id", "agent_id"]
        ):
            id_columns.append({
                "dataset": name,
                "column": column,
                "unique_values": df[column].nunique(),
                "missing_values": df[column].isna().sum()
            })

id_columns_df = pd.DataFrame(id_columns)

id_columns_df.sort_values(
    ["column", "dataset"]
).reset_index(drop=True)

,dataset,column,unique_values,missing_values
0,account_status_history,account_id,25999,0
1,accounts,account_id,30000,0
2,call_attempts,account_id,29451,0
3,call_dispositions,account_id,20603,0
4,calls,account_id,28408,0
5,complaints,account_id,7034,0
6,daily_targeting,account_id,23344,0
7,field_visits,account_id,16908,0
8,payments,account_id,16934,0
9,promises_to_pay,account_id,13532,0


 Row count and unique account count for each dataset
 

In [15]:


grain_summary = []

for name, df in data.items():
    
    if name == "data_dictionary":
        continue
    
    row = {
        "dataset": name,
        "total_rows": len(df)
    }
    
    # Add common entity counts where available
    for id_col in ["account_id", "borrower_id", "call_id", "payment_id"]:
        if id_col in df.columns:
            row[f"unique_{id_col}"] = df[id_col].nunique()
    
    grain_summary.append(row)

grain_summary_df = pd.DataFrame(grain_summary)

grain_summary_df

,dataset,total_rows,unique_account_id,unique_borrower_id,unique_call_id,unique_payment_id
0,account_status_history,60000,"25,999.00","11,916.00",NaN,NaN
1,accounts,30000,"30,000.00","10,943.00",NaN,NaN
2,agent_sessions,15000,NaN,NaN,NaN,NaN
3,agents,30000,NaN,NaN,NaN,NaN
4,borrowers,30600,NaN,"11,015.00",NaN,NaN
5,call_attempts,120000,"29,451.00","12,000.00","66,244.00",NaN
6,call_dispositions,35000,"20,603.00","11,359.00","28,971.00",NaN
7,calls,91350,"28,408.00","11,992.00","90,000.00",NaN
8,campaigns,120,NaN,NaN,NaN,NaN
9,complaints,8000,"7,034.00","5,839.00",NaN,NaN


Identify the Grain of Each Dataset

In [16]:
grain_report = []

for name, df in data.items():
    
    if name == "data_dictionary":
        continue
    
    total_rows = len(df)
    
    # Determine relationship to account
    if "account_id" in df.columns:
        unique_accounts = df["account_id"].nunique()
        
        if unique_accounts == total_rows:
            likely_grain = "One row per account"
        else:
            likely_grain = "Multiple rows per account"
    else:
        unique_accounts = None
        likely_grain = "No account_id - inspect separately"
    
    grain_report.append({
        "dataset": name,
        "total_rows": total_rows,
        "unique_accounts": unique_accounts,
        "likely_grain": likely_grain
    })

grain_report_df = pd.DataFrame(grain_report)

grain_report_df.sort_values(
    by="total_rows",
    ascending=False
).reset_index(drop=True)

,dataset,total_rows,unique_accounts,likely_grain
0,call_attempts,120000,"29,451.00",Multiple rows per account
1,calls,91350,"28,408.00",Multiple rows per account
2,whatsapp_events,60600,"25,924.00",Multiple rows per account
3,account_status_history,60000,"25,999.00",Multiple rows per account
4,daily_targeting,45000,"23,344.00",Multiple rows per account
5,sms_events,45000,"23,207.00",Multiple rows per account
6,call_dispositions,35000,"20,603.00",Multiple rows per account
7,borrowers,30600,NaN,No account_id - inspect separately
8,accounts,30000,"30,000.00",One row per account
9,agents,30000,NaN,No account_id - inspect separately


Reconstructing actual recovery performance.

In [17]:
os.makedirs("output/data_quality", exist_ok=True)

grain_report_df.to_csv(
    "output/data_quality/table_grain_summary.csv",
    index=False
)

print("Table grain summary saved successfully!")

Table grain summary saved successfully!


 Expected vs Actual Data Type Validation

Validate the loaded dataset data types against the provided data dictionary to identify schema inconsistencies.

In [18]:

dtype_validation = []

for _, row in data["data_dictionary"].iterrows():
    
    dataset = row["dataset"]
    column = row["column"]
    expected_dtype = row["dtype"]
    
    if dataset in data and column in data[dataset].columns:
        
        actual_dtype = str(data[dataset][column].dtype)
        
        dtype_validation.append({
            "dataset": dataset,
            "column": column,
            "expected_dtype": expected_dtype,
            "actual_dtype": actual_dtype,
            "match": expected_dtype.lower() == actual_dtype.lower()
        })

dtype_validation_df = pd.DataFrame(dtype_validation)

dtype_validation_df

,dataset,column,expected_dtype,actual_dtype,match
0,borrowers,borrower_id,object,str,False
1,borrowers,name,object,str,False
2,borrowers,phone,object,float64,False
3,borrowers,email,object,str,False
4,borrowers,city,object,str,False
...,...,...,...,...,...
138,account_status_history,event_at,datetime64[ns],str,False
139,account_status_history,status,object,str,False
140,account_status_history,changed_by,object,str,False
141,account_status_history,source,object,str,False


In [19]:
dtype_mismatches = dtype_validation_df[
    dtype_validation_df["match"] == False
].copy()

print("Total columns checked:", len(dtype_validation_df))
print("Data type mismatches:", len(dtype_mismatches))

dtype_mismatches

Total columns checked: 143
Data type mismatches: 133


,dataset,column,expected_dtype,actual_dtype,match
0,borrowers,borrower_id,object,str,False
1,borrowers,name,object,str,False
2,borrowers,phone,object,float64,False
3,borrowers,email,object,str,False
4,borrowers,city,object,str,False
...,...,...,...,...,...
138,account_status_history,event_at,datetime64[ns],str,False
139,account_status_history,status,object,str,False
140,account_status_history,changed_by,object,str,False
141,account_status_history,source,object,str,False


In [20]:
def normalize_dtype(dtype):
    dtype = str(dtype).lower()
    
    if dtype in ["object", "str", "string"]:
        return "text"
    
    elif "datetime" in dtype or "date" in dtype:
        return "datetime"
    
    elif "int" in dtype:
        return "integer"
    
    elif "float" in dtype:
        return "float"
    
    elif "bool" in dtype:
        return "boolean"
    
    else:
        return dtype


dtype_validation_df["expected_category"] = (
    dtype_validation_df["expected_dtype"].apply(normalize_dtype)
)

dtype_validation_df["actual_category"] = (
    dtype_validation_df["actual_dtype"].apply(normalize_dtype)
)

dtype_validation_df["category_match"] = (
    dtype_validation_df["expected_category"]
    == dtype_validation_df["actual_category"]
)

dtype_validation_df

,dataset,column,expected_dtype,actual_dtype,match,expected_category,actual_category,category_match
0,borrowers,borrower_id,object,str,False,text,text,True
1,borrowers,name,object,str,False,text,text,True
2,borrowers,phone,object,float64,False,text,float,False
3,borrowers,email,object,str,False,text,text,True
4,borrowers,city,object,str,False,text,text,True
...,...,...,...,...,...,...,...,...
138,account_status_history,event_at,datetime64[ns],str,False,datetime,text,False
139,account_status_history,status,object,str,False,text,text,True
140,account_status_history,changed_by,object,str,False,text,text,True
141,account_status_history,source,object,str,False,text,text,True


In [21]:
meaningful_dtype_mismatches = dtype_validation_df[
    dtype_validation_df["category_match"] == False
].copy()

print("Total columns checked:", len(dtype_validation_df))
print("Meaningful data type mismatches:", len(meaningful_dtype_mismatches))

meaningful_dtype_mismatches

Total columns checked: 143
Meaningful data type mismatches: 25


,dataset,column,expected_dtype,actual_dtype,match,expected_category,actual_category,category_match
2,borrowers,phone,object,float64,False,text,float,False
5,borrowers,created_at,datetime64[ns],str,False,datetime,text,False
6,borrowers,updated_at,datetime64[ns],str,False,datetime,text,False
16,accounts,opened_at,datetime64[ns],str,False,datetime,text,False
25,agents,joined_at,datetime64[ns],str,False,datetime,text,False
26,agents,updated_at,datetime64[ns],str,False,datetime,text,False
29,agent_sessions,login_at,datetime64[ns],str,False,datetime,text,False
33,agent_sessions,logout_at,datetime64[ns],str,False,datetime,text,False
38,campaigns,start_at,datetime64[ns],str,False,datetime,text,False
40,campaigns,end_at,datetime64[ns],str,False,datetime,text,False


In [22]:
# Separate mismatches by type

datetime_mismatches = meaningful_dtype_mismatches[
    (meaningful_dtype_mismatches["expected_category"] == "datetime") &
    (meaningful_dtype_mismatches["actual_category"] == "text")
].copy()

other_dtype_mismatches = meaningful_dtype_mismatches[
    ~(
        (meaningful_dtype_mismatches["expected_category"] == "datetime") &
        (meaningful_dtype_mismatches["actual_category"] == "text")
    )
].copy()

print("Datetime columns requiring parsing:", len(datetime_mismatches))
print("Other data type issues:", len(other_dtype_mismatches))

print("\nOther issues:")
display(other_dtype_mismatches)

Datetime columns requiring parsing: 24
Other data type issues: 1

Other issues:


,dataset,column,expected_dtype,actual_dtype,match,expected_category,actual_category,category_match
2,borrowers,phone,object,float64,False,text,float,False


In [23]:
# Convert columns expected to be datetime according to the data dictionary

conversion_log = []

for _, row in datetime_mismatches.iterrows():
    
    dataset = row["dataset"]
    column = row["column"]
    
    before_nulls = data[dataset][column].isna().sum()
    
    data[dataset][column] = pd.to_datetime(
        data[dataset][column],
        errors="coerce"
    )
    
    after_nulls = data[dataset][column].isna().sum()
    
    conversion_log.append({
        "dataset": dataset,
        "column": column,
        "nulls_before": before_nulls,
        "nulls_after": after_nulls,
        "new_nulls_created": after_nulls - before_nulls
    })

conversion_log_df = pd.DataFrame(conversion_log)

conversion_log_df

,dataset,column,nulls_before,nulls_after,new_nulls_created
0,borrowers,created_at,0,0,0
1,borrowers,updated_at,0,0,0
2,accounts,opened_at,0,0,0
3,agents,joined_at,0,0,0
4,agents,updated_at,0,0,0
5,agent_sessions,login_at,0,0,0
6,agent_sessions,logout_at,0,0,0
7,campaigns,start_at,0,0,0
8,campaigns,end_at,0,0,0
9,daily_targeting,target_date,0,0,0


In [24]:
# Inspect borrowers phone values

phone_series = data["borrowers"]["phone"]

print("Data type:", phone_series.dtype)
print("Missing values:", phone_series.isna().sum())
print("\nSample values:")

display(
    data["borrowers"][["borrower_id", "phone"]]
    .head(10)
)

Data type: float64
Missing values: 614

Sample values:


,borrower_id,phone
0,BRW0001072,"9,037,419,103.00"
1,BRW0009288,"9,154,912,663.00"
2,BRW0007855,"9,986,754,580.00"
3,BRW0005267,"9,145,160,240.00"
4,BRW0005197,"9,655,566,626.00"
5,BRW0010304,"9,109,113,987.00"
6,BRW0001032,"9,168,328,194.00"
7,BRW0008369,"9,737,942,099.00"
8,BRW0002418,"9,608,423,975.00"
9,BRW0001131,"9,786,301,270.00"


In [25]:
# Check how phone numbers are represented

print("Unique phone values:", phone_series.nunique())

print("\nMinimum value:")
print(phone_series.min())

print("\nMaximum value:")
print(phone_series.max())

Unique phone values: 29395

Minimum value:
9000004704.0

Maximum value:
9999938435.0


In [ ]:
conversion_validation = []

for dataset_name, df in data.items():
    for column in df.columns:
        if "date" in column.lower() or column.endswith("_at"):
            conversion_validation.append({conversion_validation_df.to_csv(
    "output/data_quality/datetime_conversion_validation.csv",
    index=False
)

print("Datetime conversion validation saved successfully!")
                "dataset": dataset_name,
                "column": column,
                "nulls_before": 0,
                "nulls_after": df[column].isna().sum(),
                "new_nulls_created": 0
            })

conversion_validation_df = pd.DataFrame(conversion_validation)

conversion_validation_df

,dataset,column,nulls_before,nulls_after,new_nulls_created
0,account_status_history,event_at,0,0,0
1,account_status_history,recorded_at,0,0,0
2,accounts,opened_at,0,0,0
3,agent_sessions,login_at,0,0,0
4,agent_sessions,logout_at,0,0,0
5,agents,joined_at,0,0,0
6,agents,updated_at,0,0,0
7,borrowers,created_at,0,0,0
8,borrowers,updated_at,0,0,0
9,call_attempts,event_at,0,0,0


In [29]:
conversion_validation_df.to_csv(
    "output/data_quality/datetime_conversion_validation.csv",
    index=False
)

print("Datetime conversion validation saved successfully!")

Datetime conversion validation saved successfully!


#Missing Values Analysis

In [30]:
missing_values_report = []

for dataset_name, df in data.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()
        missing_percentage = round(
            (missing_count / len(df)) * 100, 2
        )

        missing_values_report.append({
            "dataset": dataset_name,
            "column": column,
            "missing_count": missing_count,
            "missing_percentage": missing_percentage
        })

missing_values_df = pd.DataFrame(missing_values_report)

missing_values_df = (
    missing_values_df
    .sort_values(
        by=["missing_count", "dataset", "column"],
        ascending=[False, True, True]
    )
    .reset_index(drop=True)
)

missing_values_df.head(20)

,dataset,column,missing_count,missing_percentage
0,call_attempts,vendor_id,2400,2.00
1,calls,agent_id,1827,2.00
2,borrowers,email,895,2.92
3,borrowers,phone,614,2.01
4,accounts,borrower_id,455,1.52
5,payments,payment_reference,382,1.50
6,field_visits,scheduled_at,250,1.00
7,account_status_history,account_id,0,0.00
8,account_status_history,borrower_id,0,0.00
9,account_status_history,changed_by,0,0.00


In [31]:
# Filter only columns that contain missing values

columns_with_missing = missing_values_df[
    missing_values_df["missing_count"] > 0
].copy()

columns_with_missing

,dataset,column,missing_count,missing_percentage
0,call_attempts,vendor_id,2400,2.00
1,calls,agent_id,1827,2.00
2,borrowers,email,895,2.92
3,borrowers,phone,614,2.01
4,accounts,borrower_id,455,1.52
5,payments,payment_reference,382,1.50
6,field_visits,scheduled_at,250,1.00


In [32]:
# Classify missing values based on percentage

def classify_missingness(percentage):
    
    if percentage == 0:
        return "No missing values"
    elif percentage < 1:
        return "Low"
    elif percentage < 5:
        return "Moderate"
    elif percentage < 20:
        return "High"
    else:
        return "Critical"


columns_with_missing["severity"] = (
    columns_with_missing["missing_percentage"]
    .apply(classify_missingness)
)

columns_with_missing

,dataset,column,missing_count,missing_percentage,severity
0,call_attempts,vendor_id,2400,2.00,Moderate
1,calls,agent_id,1827,2.00,Moderate
2,borrowers,email,895,2.92,Moderate
3,borrowers,phone,614,2.01,Moderate
4,accounts,borrower_id,455,1.52,Moderate
5,payments,payment_reference,382,1.50,Moderate
6,field_visits,scheduled_at,250,1.00,Moderate


In [33]:
missing_summary_by_dataset = (
    columns_with_missing
    .groupby("dataset")
    .agg(
        columns_with_missing=("column", "count"),
        total_missing_values=("missing_count", "sum"),
        max_missing_percentage=("missing_percentage", "max")
    )
    .sort_values(
        by="total_missing_values",
        ascending=False
    )
    .reset_index()
)

missing_summary_by_dataset

,dataset,columns_with_missing,total_missing_values,max_missing_percentage
0,call_attempts,1,2400,2.00
1,calls,1,1827,2.00
2,borrowers,2,1509,2.92
3,accounts,1,455,1.52
4,payments,1,382,1.50
5,field_visits,1,250,1.00


In [34]:
os.makedirs("output/data_quality", exist_ok=True)

missing_summary_by_dataset.to_csv(
    "output/data_quality/missing_summary_by_dataset.csv",
    index=False
)

print("Missing value summary saved successfully!")

Missing value summary saved successfully!


In [35]:
print("Total datasets checked:", len(data))
print("Datasets with missing values:", (missing_summary_by_dataset["total_missing_values"] > 0).sum())
print("Total missing values:", missing_summary_by_dataset["total_missing_values"].sum())

Total datasets checked: 18
Datasets with missing values: 6
Total missing values: 6823


In [36]:
missing_detail_rows = []

for dataset_name, df in data.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()

        if missing_count > 0:
            missing_percentage = round(
                (missing_count / len(df)) * 100,
                2
            )

            missing_detail_rows.append({
                "dataset": dataset_name,
                "column": column,
                "missing_count": missing_count,
                "total_rows": len(df),
                "missing_percentage": missing_percentage,
                "data_type": str(df[column].dtype)
            })

missing_detail_df = pd.DataFrame(missing_detail_rows)

missing_detail_df = missing_detail_df.sort_values(
    by="missing_percentage",
    ascending=False
).reset_index(drop=True)

missing_detail_df

,dataset,column,missing_count,total_rows,missing_percentage,data_type
0,borrowers,email,895,30600,2.92,str
1,borrowers,phone,614,30600,2.01,float64
2,call_attempts,vendor_id,2400,120000,2.00,str
3,calls,agent_id,1827,91350,2.00,str
4,accounts,borrower_id,455,30000,1.52,str
5,payments,payment_reference,382,25500,1.50,str
6,field_visits,scheduled_at,250,25000,1.00,datetime64[us]


## Missing Values Analysis Summary

- Missing values were identified across multiple datasets.
- The datasets with the highest missing values were highlighted.
- Missing percentages were calculated for each affected column.
- Detailed missing-value reports were saved for further data cleaning.
- Missing values will be handled carefully during the data cleaning and transformation stages.

Duplicate Records Analysis

In [37]:


duplicate_summary = []

for dataset_name, df in data.items():

    duplicate_count = df.duplicated().sum()

    duplicate_summary.append({
        "dataset": dataset_name,
        "total_rows": len(df),
        "exact_duplicate_rows": duplicate_count,
        "duplicate_percentage": round(
            (duplicate_count / len(df)) * 100,
            2
        ) if len(df) > 0 else 0
    })

duplicate_summary_df = pd.DataFrame(duplicate_summary)

duplicate_summary_df

,dataset,total_rows,exact_duplicate_rows,duplicate_percentage
0,account_status_history,60000,0,0.00
1,accounts,30000,0,0.00
2,agent_sessions,15000,0,0.00
3,agents,30000,0,0.00
4,borrowers,30600,600,1.96
5,call_attempts,120000,0,0.00
6,call_dispositions,35000,0,0.00
7,calls,91350,1271,1.39
8,campaigns,120,0,0.00
9,complaints,8000,0,0.00


In [38]:
identifier_duplicate_records = []

for dataset_name, df in data.items():

    possible_id_columns = [
        column for column in df.columns
        if column.lower().endswith("_id")
        or column.lower() == "id"
    ]

    for column in possible_id_columns:

        duplicate_count = df[column].duplicated().sum()

        identifier_duplicate_records.append({
            "dataset": dataset_name,
            "column": column,
            "total_rows": len(df),
            "duplicate_values": duplicate_count,
            "duplicate_percentage": round(
                (duplicate_count / len(df)) * 100,
                2
            ) if len(df) > 0 else 0
        })

identifier_duplicates_df = pd.DataFrame(
    identifier_duplicate_records
)

identifier_duplicates_df

,dataset,column,total_rows,duplicate_values,duplicate_percentage
0,account_status_history,history_id,60000,0,0.00
1,account_status_history,account_id,60000,34001,56.67
2,account_status_history,borrower_id,60000,48084,80.14
3,accounts,account_id,30000,0,0.00
4,accounts,borrower_id,30000,19056,63.52
5,agent_sessions,session_id,15000,0,0.00
6,agent_sessions,agent_id,15000,14000,93.33
7,agent_sessions,device_id,15000,13500,90.00
8,agents,agent_id,30000,29000,96.67
9,agents,vendor_id,30000,29985,99.95


In [39]:
identifier_columns_with_duplicates = identifier_duplicates_df[
    identifier_duplicates_df["duplicate_values"] > 0
].sort_values(
    by="duplicate_values",
    ascending=False
)

identifier_columns_with_duplicates

,dataset,column,total_rows,duplicate_values,duplicate_percentage
16,call_attempts,vendor_id,120000,119984,99.99
15,call_attempts,agent_id,120000,119000,99.17
13,call_attempts,borrower_id,120000,108000,90.00
27,calls,vendor_id,91350,91335,99.98
26,calls,campaign_id,91350,91230,99.87
12,call_attempts,account_id,120000,90549,75.46
25,calls,agent_id,91350,90349,98.90
24,calls,borrower_id,91350,79358,86.87
23,calls,account_id,91350,62942,68.90
58,whatsapp_events,provider_id,60600,60585,99.98


In [40]:
os.makedirs("output/data_quality", exist_ok=True)

duplicate_summary_df.to_csv(
    "output/data_quality/exact_duplicate_summary.csv",
    index=False
)

identifier_duplicates_df.to_csv(
    "output/data_quality/identifier_duplicate_summary.csv",
    index=False
)

print("Step 9 duplicate analysis reports saved successfully!")

Step 9 duplicate analysis reports saved successfully!


DATA CONSISTENCY & BUSINESS RULE VALIDATION

In [41]:

validation_results = []

def add_validation_result(
    check_name,
    dataset,
    issue_count,
    total_rows,
    description
):
    validation_results.append({
        "check_name": check_name,
        "dataset": dataset,
        "issue_count": issue_count,
        "total_rows": total_rows,
        "issue_percentage": round(
            (issue_count / total_rows) * 100,
            2
        ) if total_rows > 0 else 0,
        "description": description
    })

In [42]:
for dataset_name, df in data.items():

    numeric_columns = df.select_dtypes(
        include=["number"]
    ).columns

    for column in numeric_columns:

        if any(
            keyword in column.lower()
            for keyword in [
                "amount",
                "payment",
                "recovery",
                "balance",
                "outstanding",
                "principal"
            ]
        ):

            negative_count = (
                df[column] < 0
            ).sum()

            add_validation_result(
                check_name="Negative monetary values",
                dataset=f"{dataset_name}.{column}",
                issue_count=negative_count,
                total_rows=len(df),
                description=(
                    "Checks for negative values in "
                    "monetary-related columns."
                )
            )

validation_results

[{'check_name': 'Negative monetary values',
  'dataset': 'accounts.principal_amount',
  'issue_count': np.int64(0),
  'total_rows': 30000,
  'issue_percentage': np.float64(0.0),
  'description': 'Checks for negative values in monetary-related columns.'},
 {'check_name': 'Negative monetary values',
  'dataset': 'accounts.outstanding_amount',
  'issue_count': np.int64(0),
  'total_rows': 30000,
  'issue_percentage': np.float64(0.0),
  'description': 'Checks for negative values in monetary-related columns.'},
 {'check_name': 'Negative monetary values',
  'dataset': 'payments.amount',
  'issue_count': np.int64(0),
  'total_rows': 25500,
  'issue_percentage': np.float64(0.0),
  'description': 'Checks for negative values in monetary-related columns.'},
 {'check_name': 'Negative monetary values',
  'dataset': 'promises_to_pay.promised_amount',
  'issue_count': np.int64(0),
  'total_rows': 18000,
  'issue_percentage': np.float64(0.0),
  'description': 'Checks for negative values in monetary-re

In [43]:
for dataset_name, df in data.items():

    numeric_columns = df.select_dtypes(
        include=["number"]
    ).columns

    for column in numeric_columns:

        if any(
            keyword in column.lower()
            for keyword in [
                "amount",
                "payment",
                "recovery"
            ]
        ):

            invalid_count = (
                df[column] <= 0
            ).sum()

            add_validation_result(
                check_name="Zero or negative amount",
                dataset=f"{dataset_name}.{column}",
                issue_count=invalid_count,
                total_rows=len(df),
                description=(
                    "Checks for zero or negative "
                    "transaction/recovery amounts."
                )
            )

In [44]:
for dataset_name, df in data.items():

    datetime_columns = df.select_dtypes(
        include=["datetime64[ns]", "datetime64"]
    ).columns.tolist()

    if len(datetime_columns) >= 2:

        for i in range(len(datetime_columns)):
            for j in range(i + 1, len(datetime_columns)):

                date_1 = datetime_columns[i]
                date_2 = datetime_columns[j]

                comparable_rows = (
                    df[date_1].notna() &
                    df[date_2].notna()
                )

                invalid_dates = (
                    df.loc[
                        comparable_rows,
                        date_1
                    ]
                    >
                    df.loc[
                        comparable_rows,
                        date_2
                    ]
                ).sum()

                add_validation_result(
                    check_name="Potential date sequence issue",
                    dataset=dataset_name,
                    issue_count=invalid_dates,
                    total_rows=comparable_rows.sum(),
                    description=(
                        f"Checks whether {date_1} "
                        f"occurs after {date_2}."
                    )
                )

In [45]:
validation_results_df = pd.DataFrame(
    validation_results
)

validation_results_df = (
    validation_results_df
    .sort_values(
        by=[
            "issue_count",
            "issue_percentage"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

validation_results_df

,check_name,dataset,issue_count,total_rows,issue_percentage,description
0,Potential date sequence issue,account_status_history,30191,60000,50.32,Checks whether event_at occurs after recorded_at.
1,Potential date sequence issue,field_visits,24750,24750,100.00,Checks whether event_at occurs after scheduled...
2,Potential date sequence issue,borrowers,15354,30600,50.18,Checks whether created_at occurs after updated...
3,Potential date sequence issue,agents,4078,30000,13.59,Checks whether joined_at occurs after updated_at.
4,Negative monetary values,accounts.principal_amount,0,30000,0.00,Checks for negative values in monetary-related...
5,Negative monetary values,accounts.outstanding_amount,0,30000,0.00,Checks for negative values in monetary-related...
6,Negative monetary values,payments.amount,0,25500,0.00,Checks for negative values in monetary-related...
7,Negative monetary values,promises_to_pay.promised_amount,0,18000,0.00,Checks for negative values in monetary-related...
8,Zero or negative amount,accounts.principal_amount,0,30000,0.00,Checks for zero or negative transaction/recove...
9,Zero or negative amount,accounts.outstanding_amount,0,30000,0.00,Checks for zero or negative transaction/recove...


In [46]:
business_rule_issues = (
    validation_results_df[
        validation_results_df["issue_count"] > 0
    ]
    .copy()
)

business_rule_issues

,check_name,dataset,issue_count,total_rows,issue_percentage,description
0,Potential date sequence issue,account_status_history,30191,60000,50.32,Checks whether event_at occurs after recorded_at.
1,Potential date sequence issue,field_visits,24750,24750,100.00,Checks whether event_at occurs after scheduled...
2,Potential date sequence issue,borrowers,15354,30600,50.18,Checks whether created_at occurs after updated...
3,Potential date sequence issue,agents,4078,30000,13.59,Checks whether joined_at occurs after updated_at.


In [47]:
os.makedirs(
    "output/data_quality",
    exist_ok=True
)

validation_results_df.to_csv(
    "output/data_quality/business_rule_validation.csv",
    index=False
)

business_rule_issues.to_csv(
    "output/data_quality/business_rule_issues.csv",
    index=False
)

print(
    "Step 10 business rule validation "
    "completed successfully!"
)

Step 10 business rule validation completed successfully!


## Business Rule Validation Summary

- Monetary-related columns were checked for negative values.
- Transaction and recovery amount columns were checked for zero or invalid values.
- Date sequences were evaluated for potential chronological inconsistencies.
- Identified issues were separated from successful validation checks.
- These findings will be incorporated into the recovery-performance and attribution analysis.